Casus: Ter voorbereiding van een machine learning project voor de voorspelling van kijkcijfers voor Vlaamse televisieprogramma's zijn drie databronnen nodig: 
- de gegevens van de kijkcijfers zelf, gescrapet van de website van de cim. Zie bijv. https://www.cim.be/nl/televisie?type=daily&date=2026-4-9&region=north. In de praktijk zal gebruik gemaakt worden van de API van de website: 
- de omzetting van de combinatie (zender, programmanaam) naar een bepaalde programmagenre. Hiervoor zal gebruik gemaakt worden van de API van de LLM gemini (Google)
- de weergegevens (neerslag en temperatuur)

Onderstaand stukje code toont hoe de kijkcijfers voor één dag (vandaag -30 dagen) kunnen opgehaald worden. 

### Opgave 1
Integreer dit stuk code (en pas indien nodig aan) in de functie KijkCijfers::scrape_data om de kijkcijfers voor de volledige in de functie main() opgegeven periode op te halen. 

In [6]:
import datetime
import pytz
import time
import os
import sys
import requests
import json
import pandas as pd
pd.set_option("display.width", 140)

current_date = datetime.date.today() - datetime.timedelta(days=30)
url = "https://api.cim.be/api/cim_tv_public_results_daily_views?dateDiff={date}&reportType=north"
year = current_date.year
date_str = current_date.strftime("%Y-%m-%d")
current_url = url.format(date=date_str)
records = []
print(f"Fetching data for {date_str}...")
try:
    response = requests.get(current_url)
    if response.status_code == 200:
        data = response.json()
        data = data.get('hydra:member')
        for item in data:
            record = {
                'ranking': item.get('ranking'),
                'description': item.get('description').upper(),
                'channel': item.get('channel').upper(),
                'dateDiff': item.get('dateDiff'),
                'startTime': item.get('startTime'),
                'rLength': item.get('rLength'),
                'rateInk': item.get('rateInK')
            }
            for key in record:
                val = record[key]
                if type(val) == str:
                    record[key] = val.replace('"', '')  # remove double quotes

            if len(record['description']) > 0: # only keep records with a description
                records.append(record)
    else:
        print(f"Request failed with status code {response.status_code}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred: {e}")

df = pd.DataFrame(records)
print(df.head(len(df)))  # print all records to check the data

Fetching data for 2026-03-12...
   ranking                                        description     channel                    dateDiff startTime   rLength  rateInk
0        1                                 HET 7 UUR-JOURNAAL       VRT 1  2026-03-12T00:00:00.000000  19:00:00  00:44:50  853.248
1        2                                              THUIS       VRT 1  2026-03-12T00:00:00.000000  20:16:07  00:23:05  808.081
2        3                                     NIEUWS 19U VTM         VTM  2026-03-12T00:00:00.000000  18:59:44  00:57:11  601.032
3        4                                      MAN BIJT HOND       VRT 1  2026-03-12T00:00:00.000000  19:47:07  00:20:04  594.011
4        5                                            BLOKKEN       VRT 1  2026-03-12T00:00:00.000000  18:28:11  00:28:28  555.002
5        6                                            FAMILIE         VTM  2026-03-12T00:00:00.000000  20:10:03  00:24:22  464.866
6        7                                 DE DAG V

### Opgave 2

- Gaan naar https://openmeteo.com en haal de Python-code op om de historische weerdata (temperatuur 2m en neerslag) op te halen voor een bepaalde periode op je actuele locatie. 

- Integreer deze code in de functie Weather::get_weather()

- Zorg ervoor dat je tot 5 maal probeert als de API niet beschikbaar is of andere fouten geeft. Wacht telkens 5 minuten tussen 2 pogingen. Hoe ga je dit testen? 

### Opgave 3

Ontwerp een klasse om te loggen. 
- Denk na over wat in de contructor kan komen en wat in member-functies
- Schrijf alle foutmeldingen en eventuele info-boodschappen weg in een tekstbestand met timestamp, ernst van de fout (info, error, ...) en foutboodschap. 
- Vervang alle printboodschappen door een logging. 
- Denk na over waar nog extra logging kan bij komen. 